# Codex Python SDK Walkthrough

Public SDK surface only (`codex_app_server` root exports).

In [ ]:
# Cell 1: bootstrap current kernel with local SDK + dependencies
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(
        f'Notebook requires Python 3.10+; current interpreter is {sys.version.split()[0]}.'
    )

try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(Path.home()))

candidates = [
    Path.cwd(),
    Path.home() / '.openclaw/workspace/codex-py-sdk/sdk/python',
]
repo_python_dir = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if repo_python_dir is None:
    raise RuntimeError('Could not locate sdk/python. Set repo_python_dir manually.')

src_dir = repo_python_dir / 'src'
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_python_dir)])
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        'Editable install failed for this kernel environment. Upgrade pip in this environment '
        '(for example: `python -m pip install --upgrade pip`) and rerun this cell.'
    ) from exc

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Force fresh imports after SDK upgrades in the same notebook kernel.
for module_name in list(sys.modules):
    if module_name == 'codex_app_server' or module_name.startswith('codex_app_server.'):
        sys.modules.pop(module_name, None)

print('Kernel:', sys.executable)
print('SDK source:', src_dir)


In [ ]:
# Cell 2: imports (public only)
from codex_app_server import (
    AsyncCodex,
    Codex,
    ImageInput,
    LocalImageInput,
    TextInput,
    retry_on_overload,
)

In [ ]:
# Cell 3: simple sync conversation
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
    turn = thread.turn(TextInput('Explain gradient descent in 3 bullets.'))
    result = turn.run()

    print('server:', codex.metadata)
    print('status:', result.status)
    print(result.text)

In [ ]:
# Cell 4: multi-turn continuity in same thread
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})

    first = thread.turn(TextInput('Give a short summary of transformers.')).run()
    second = thread.turn(TextInput('Now explain that to a high-school student.')).run()

    print('first status:', first.status)
    print('second status:', second.status)
    print('second text:', second.text)

In [ ]:
# Cell 5: full thread lifecycle and branching (sync)
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
    first = thread.turn(TextInput('One sentence about structured planning.')).run()
    second = thread.turn(TextInput('Now restate it for a junior engineer.')).run()

    reopened = codex.thread(thread.id)
    listing_active = codex.thread_list(limit=20, archived=False)
    reading = reopened.read(include_turns=True)

    _ = reopened.set_name('sdk-lifecycle-demo')
    _ = reopened.archive()
    listing_archived = codex.thread_list(limit=20, archived=True)
    unarchived = reopened.unarchive()

    resumed_info = 'n/a'
    try:
        resumed = unarchived.resume(model='gpt-5', config={'model_reasoning_effort': 'high'})
        resumed_result = resumed.turn(TextInput('Continue in one short sentence.')).run()
        resumed_info = f'{resumed_result.turn_id} {resumed_result.status}'
    except Exception as e:
        resumed_info = f'skipped({type(e).__name__})'

    forked_info = 'n/a'
    try:
        forked = unarchived.fork(model='gpt-5')
        forked_result = forked.turn(TextInput('Take a different angle in one short sentence.')).run()
        forked_info = f'{forked_result.turn_id} {forked_result.status}'
    except Exception as e:
        forked_info = f'skipped({type(e).__name__})'

    compact_info = 'sent'
    try:
        _ = unarchived.compact()
    except Exception as e:
        compact_info = f'skipped({type(e).__name__})'

    print('Lifecycle OK:', thread.id)
    print('first:', first.turn_id, first.status)
    print('second:', second.turn_id, second.status)
    print('read.turns:', len(reading.thread.turns or []))
    print('list.active:', len(listing_active.data))
    print('list.archived:', len(listing_archived.data))
    print('resumed:', resumed_info)
    print('forked:', forked_info)
    print('compact:', compact_info)



In [ ]:
# Cell 6: multimodal with remote image
remote_image_url = 'https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png'

with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
    result = thread.turn([
        TextInput('What do you see in this image? 3 bullets.'),
        ImageInput(remote_image_url),
    ]).run()

    print('status:', result.status)
    print(result.text)


In [ ]:
# Cell 7: multimodal with local image (bundled asset)
local_image_path = repo_python_dir / 'examples' / 'assets' / 'sample_scene.png'
if not local_image_path.exists():
    raise FileNotFoundError(f'Missing bundled image: {local_image_path}')

with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
    result = thread.turn([
        TextInput('Describe this local image in 2 bullets.'),
        LocalImageInput(str(local_image_path.resolve())),
    ]).run()

    print('status:', result.status)
    print(result.text)


In [ ]:
# Cell 8: retry-on-overload pattern
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})

    result = retry_on_overload(
        lambda: thread.turn(TextInput('List 5 failure modes in distributed systems.')).run(),
        max_attempts=3,
        initial_delay_s=0.25,
        max_delay_s=2.0,
    )

    print('status:', result.status)
    print(result.text)

In [ ]:
# Cell 9: full thread lifecycle and branching (async)
import asyncio


async def async_lifecycle_demo():
    async with AsyncCodex() as codex:
        thread = await codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
        first = await (await thread.turn(TextInput('One sentence about structured planning.'))).run()
        second = await (await thread.turn(TextInput('Now restate it for a junior engineer.'))).run()

        reopened = codex.thread(thread.id)
        listing_active = await codex.thread_list(limit=20, archived=False)
        reading = await reopened.read(include_turns=True)

        _ = await reopened.set_name('sdk-lifecycle-demo')
        _ = await reopened.archive()
        listing_archived = await codex.thread_list(limit=20, archived=True)
        unarchived = await reopened.unarchive()

        resumed_info = 'n/a'
        try:
            resumed = await unarchived.resume(model='gpt-5', config={'model_reasoning_effort': 'high'})
            resumed_result = await (await resumed.turn(TextInput('Continue in one short sentence.'))).run()
            resumed_info = f'{resumed_result.turn_id} {resumed_result.status}'
        except Exception as e:
            resumed_info = f'skipped({type(e).__name__})'

        forked_info = 'n/a'
        try:
            forked = await unarchived.fork(model='gpt-5')
            forked_result = await (await forked.turn(TextInput('Take a different angle in one short sentence.'))).run()
            forked_info = f'{forked_result.turn_id} {forked_result.status}'
        except Exception as e:
            forked_info = f'skipped({type(e).__name__})'

        compact_info = 'sent'
        try:
            _ = await unarchived.compact()
        except Exception as e:
            compact_info = f'skipped({type(e).__name__})'

        print('Lifecycle OK:', thread.id)
        print('first:', first.turn_id, first.status)
        print('second:', second.turn_id, second.status)
        print('read.turns:', len(reading.thread.turns or []))
        print('list.active:', len(listing_active.data))
        print('list.archived:', len(listing_archived.data))
        print('resumed:', resumed_info)
        print('forked:', forked_info)
        print('compact:', compact_info)


await async_lifecycle_demo()



In [ ]:
# Cell 10: async stream + steer + interrupt (best effort)
import asyncio


async def async_stream_demo():
    async with AsyncCodex() as codex:
        thread = await codex.thread_start(model='gpt-5', config={'model_reasoning_effort': 'high'})
        turn = await thread.turn(TextInput('Count from 1 to 200 with commas, then one summary sentence.'))

        try:
            _ = await turn.steer(TextInput('Keep it brief and stop after 20 numbers.'))
            print('steer: sent')
        except Exception as e:
            print('steer: skipped', type(e).__name__)

        try:
            _ = await turn.interrupt()
            print('interrupt: sent')
        except Exception as e:
            print('interrupt: skipped', type(e).__name__)

        event_count = 0
        async for event in turn.stream():
            event_count += 1
            print(event.method, event.payload)

        print('events.count:', event_count)


await async_stream_demo()

